# System Exploration

Parsing the data and understanding it (System attribute)

In [31]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from evtx import PyEvtxParser

plt.style.use('ggplot')

In [32]:
# Path to the data
project_folder = Path.cwd().parent

evtx_path = project_folder / "data/raw/93_securitelog.evtx"
evtx_path

WindowsPath('c:/Users/hp/Documents/School/anomaly-detection/data/raw/93_securitelog.evtx')

In [33]:
# Extract the data
parser = PyEvtxParser(evtx_path)

records = [json.loads(record['data']) for record in parser.records_json()]

# Remove extra key
records = [record['Event'] for record in records]

len(records)

31466

In [34]:
# Flatten useless '#attributes'
def flatten_record(record):
    system = record['System']
    
    for key in ('Provider', 'TimeCreated', 'Correlation', 'Execution'):
        value = system.get(key)
        if value:
            system[key] = value['#attributes']
    
    return record

records_clean = [flatten_record(record) for record in records]

In [35]:
# Extract event records
event_records = [record["EventData"] for record in records_clean]

In [36]:
with open(project_folder / "outputs/record.txt", "w", encoding="utf-8") as file:
    file.write(json.dumps(
        records_clean[0],
        indent=4
    ))

output_path = project_folder / "outputs/event_exploration"

with open(output_path / "event_record.txt", "w", encoding="utf-8") as file:
    file.write(json.dumps(
        event_records[0],
        indent=4
    ))
    
event_records[0]

{'SubjectUserSid': 'S-1-5-18',
 'SubjectUserName': 'ESANTE$',
 'SubjectDomainName': 'CARTE',
 'SubjectLogonId': '0x3e7',
 'TargetUserSid': 'S-1-5-21-3333335972-2113272682-612058519-500',
 'TargetUserName': 'Administrateur',
 'TargetDomainName': 'ESANTE',
 'TargetLogonId': '0x77061c91',
 'LogonType': 4,
 'LogonProcessName': 'Advapi  ',
 'AuthenticationPackageName': 'Negotiate',
 'WorkstationName': 'ESANTE',
 'LogonGuid': '00000000-0000-0000-0000-000000000000',
 'TransmittedServices': '-',
 'LmPackageName': '-',
 'KeyLength': 0,
 'ProcessId': '0x898',
 'ProcessName': 'C:\\Windows\\System32\\svchost.exe',
 'IpAddress': '-',
 'IpPort': '-',
 'ImpersonationLevel': '%%1833',
 'RestrictedAdminMode': '-',
 'TargetOutboundUserName': '-',
 'TargetOutboundDomainName': '-',
 'VirtualAccount': '%%1843',
 'TargetLinkedLogonId': '0x0',
 'ElevatedToken': '%%1842'}

In [37]:
# Extract EventData alongside EventID for reference
event_data_records = [
    {'EventID': record['System']['EventID'], **record['EventData']}
    for record in records_clean
]

event_data_records[0]

{'EventID': 4624,
 'SubjectUserSid': 'S-1-5-18',
 'SubjectUserName': 'ESANTE$',
 'SubjectDomainName': 'CARTE',
 'SubjectLogonId': '0x3e7',
 'TargetUserSid': 'S-1-5-21-3333335972-2113272682-612058519-500',
 'TargetUserName': 'Administrateur',
 'TargetDomainName': 'ESANTE',
 'TargetLogonId': '0x77061c91',
 'LogonType': 4,
 'LogonProcessName': 'Advapi  ',
 'AuthenticationPackageName': 'Negotiate',
 'WorkstationName': 'ESANTE',
 'LogonGuid': '00000000-0000-0000-0000-000000000000',
 'TransmittedServices': '-',
 'LmPackageName': '-',
 'KeyLength': 0,
 'ProcessId': '0x898',
 'ProcessName': 'C:\\Windows\\System32\\svchost.exe',
 'IpAddress': '-',
 'IpPort': '-',
 'ImpersonationLevel': '%%1833',
 'RestrictedAdminMode': '-',
 'TargetOutboundUserName': '-',
 'TargetOutboundDomainName': '-',
 'VirtualAccount': '%%1843',
 'TargetLinkedLogonId': '0x0',
 'ElevatedToken': '%%1842'}

In [38]:
# See what fields exist across all event types
from collections import defaultdict

fields_by_eventid = defaultdict(set)
for record in event_data_records:
    eid = record['EventID']
    fields_by_eventid[eid].update(record.keys())

fields_by_eventid = dict(fields_by_eventid)

fields_by_eventid = {eid: sorted(fields) for eid, fields in fields_by_eventid.items()}

# Save it to a file
with open(output_path / "fields_by_event.txt", "w", encoding="utf-8") as file:
    file.write(json.dumps(
        fields_by_eventid,
        indent=4
    ))

fields_by_eventid[4624]

['AuthenticationPackageName',
 'ElevatedToken',
 'EventID',
 'ImpersonationLevel',
 'IpAddress',
 'IpPort',
 'KeyLength',
 'LmPackageName',
 'LogonGuid',
 'LogonProcessName',
 'LogonType',
 'ProcessId',
 'ProcessName',
 'RestrictedAdminMode',
 'SubjectDomainName',
 'SubjectLogonId',
 'SubjectUserName',
 'SubjectUserSid',
 'TargetDomainName',
 'TargetLinkedLogonId',
 'TargetLogonId',
 'TargetOutboundDomainName',
 'TargetOutboundUserName',
 'TargetUserName',
 'TargetUserSid',
 'TransmittedServices',
 'VirtualAccount',
 'WorkstationName']

# Log On Events (4624)

In [39]:
log_on_records = [record for record in event_data_records if record["EventID"] == 4624]

log_on_records[0]

{'EventID': 4624,
 'SubjectUserSid': 'S-1-5-18',
 'SubjectUserName': 'ESANTE$',
 'SubjectDomainName': 'CARTE',
 'SubjectLogonId': '0x3e7',
 'TargetUserSid': 'S-1-5-21-3333335972-2113272682-612058519-500',
 'TargetUserName': 'Administrateur',
 'TargetDomainName': 'ESANTE',
 'TargetLogonId': '0x77061c91',
 'LogonType': 4,
 'LogonProcessName': 'Advapi  ',
 'AuthenticationPackageName': 'Negotiate',
 'WorkstationName': 'ESANTE',
 'LogonGuid': '00000000-0000-0000-0000-000000000000',
 'TransmittedServices': '-',
 'LmPackageName': '-',
 'KeyLength': 0,
 'ProcessId': '0x898',
 'ProcessName': 'C:\\Windows\\System32\\svchost.exe',
 'IpAddress': '-',
 'IpPort': '-',
 'ImpersonationLevel': '%%1833',
 'RestrictedAdminMode': '-',
 'TargetOutboundUserName': '-',
 'TargetOutboundDomainName': '-',
 'VirtualAccount': '%%1843',
 'TargetLinkedLogonId': '0x0',
 'ElevatedToken': '%%1842'}

In [40]:
log_on_df = pd.DataFrame(log_on_records)

log_on_df.head()

,EventID,SubjectUserSid,SubjectUserName,SubjectDomainName,SubjectLogonId,TargetUserSid,TargetUserName,TargetDomainName,TargetLogonId,LogonType,...,ProcessName,IpAddress,IpPort,ImpersonationLevel,RestrictedAdminMode,TargetOutboundUserName,TargetOutboundDomainName,VirtualAccount,TargetLinkedLogonId,ElevatedToken
0,4624,S-1-5-18,ESANTE$,CARTE,0x3e7,S-1-5-21-3333335972-2113272682-612058519-500,Administrateur,ESANTE,0x77061c91,4,...,C:\Windows\System32\svchost.exe,-,-,%%1833,-,-,-,%%1843,0x0,%%1842
1,4624,S-1-5-18,ESANTE$,CARTE,0x3e7,S-1-5-21-3333335972-2113272682-612058519-500,Administrateur,ESANTE,0x77067629,4,...,C:\Windows\System32\svchost.exe,-,-,%%1833,-,-,-,%%1843,0x0,%%1842
2,4624,S-1-5-18,ESANTE$,CARTE,0x3e7,S-1-5-21-3333335972-2113272682-612058519-500,Administrateur,ESANTE,0x7706933c,4,...,C:\Windows\System32\svchost.exe,-,-,%%1833,-,-,-,%%1843,0x0,%%1842
3,4624,S-1-5-18,ESANTE$,CARTE,0x3e7,S-1-5-21-3333335972-2113272682-612058519-500,Administrateur,ESANTE,0x77069356,4,...,C:\Windows\System32\svchost.exe,-,-,%%1833,-,-,-,%%1843,0x0,%%1842
4,4624,S-1-5-18,ESANTE$,CARTE,0x3e7,S-1-5-21-3333335972-2113272682-612058519-500,Administrateur,ESANTE,0x7706933a,4,...,C:\Windows\System32\svchost.exe,-,-,%%1833,-,-,-,%%1843,0x0,%%1842


In [41]:
list(log_on_df.columns)

['EventID',
 'SubjectUserSid',
 'SubjectUserName',
 'SubjectDomainName',
 'SubjectLogonId',
 'TargetUserSid',
 'TargetUserName',
 'TargetDomainName',
 'TargetLogonId',
 'LogonType',
 'LogonProcessName',
 'AuthenticationPackageName',
 'WorkstationName',
 'LogonGuid',
 'TransmittedServices',
 'LmPackageName',
 'KeyLength',
 'ProcessId',
 'ProcessName',
 'IpAddress',
 'IpPort',
 'ImpersonationLevel',
 'RestrictedAdminMode',
 'TargetOutboundUserName',
 'TargetOutboundDomainName',
 'VirtualAccount',
 'TargetLinkedLogonId',
 'ElevatedToken']

In [42]:
log_on_constants = ["SubjectUserSid", "SubjectUserName", "SubjectDomainName", "SubjectLogonId", "TargetUserSid", "TargetUserName", "TargetDomainName", "LogonType", "LogonProcessName", "AuthenticationPackageName", "WorkstationName", "LogonGuid", "TransmittedServices", "LmPackageName", "KeyLength", "ProcessId", "ProcessName", "IpAddress", "IpPort", "ImpersonationLevel", "RestrictedAdminMode", "TargetOutboundUserName", "TargetOutboundDomainName", "VirtualAccount", "TargetLinkedLogonId", "ElevatedToken"]

log_on_df[log_on_constants].value_counts()

SubjectUserSid  SubjectUserName  SubjectDomainName  SubjectLogonId  TargetUserSid                                 TargetUserName  TargetDomainName  LogonType  LogonProcessName  AuthenticationPackageName  WorkstationName  LogonGuid                             TransmittedServices  LmPackageName  KeyLength  ProcessId  ProcessName                       IpAddress     IpPort  ImpersonationLevel  RestrictedAdminMode  TargetOutboundUserName  TargetOutboundDomainName  VirtualAccount  TargetLinkedLogonId  ElevatedToken
S-1-5-18        ESANTE$          CARTE              0x3e7           S-1-5-21-3333335972-2113272682-612058519-500  Administrateur  ESANTE            4          Advapi            Negotiate                  ESANTE           00000000-0000-0000-0000-000000000000  -                    -              0          0x898      C:\Windows\System32\svchost.exe   -             -       %%1833              -                    -                       -                         %%1843          0x0  

In [43]:
log_on_df[list(log_on_df.columns)].head()

,EventID,SubjectUserSid,SubjectUserName,SubjectDomainName,SubjectLogonId,TargetUserSid,TargetUserName,TargetDomainName,TargetLogonId,LogonType,...,ProcessName,IpAddress,IpPort,ImpersonationLevel,RestrictedAdminMode,TargetOutboundUserName,TargetOutboundDomainName,VirtualAccount,TargetLinkedLogonId,ElevatedToken
0,4624,S-1-5-18,ESANTE$,CARTE,0x3e7,S-1-5-21-3333335972-2113272682-612058519-500,Administrateur,ESANTE,0x77061c91,4,...,C:\Windows\System32\svchost.exe,-,-,%%1833,-,-,-,%%1843,0x0,%%1842
1,4624,S-1-5-18,ESANTE$,CARTE,0x3e7,S-1-5-21-3333335972-2113272682-612058519-500,Administrateur,ESANTE,0x77067629,4,...,C:\Windows\System32\svchost.exe,-,-,%%1833,-,-,-,%%1843,0x0,%%1842
2,4624,S-1-5-18,ESANTE$,CARTE,0x3e7,S-1-5-21-3333335972-2113272682-612058519-500,Administrateur,ESANTE,0x7706933c,4,...,C:\Windows\System32\svchost.exe,-,-,%%1833,-,-,-,%%1843,0x0,%%1842
3,4624,S-1-5-18,ESANTE$,CARTE,0x3e7,S-1-5-21-3333335972-2113272682-612058519-500,Administrateur,ESANTE,0x77069356,4,...,C:\Windows\System32\svchost.exe,-,-,%%1833,-,-,-,%%1843,0x0,%%1842
4,4624,S-1-5-18,ESANTE$,CARTE,0x3e7,S-1-5-21-3333335972-2113272682-612058519-500,Administrateur,ESANTE,0x7706933a,4,...,C:\Windows\System32\svchost.exe,-,-,%%1833,-,-,-,%%1843,0x0,%%1842
